# Metropolis-Hastings Algorithm for Discrete States

This notebook demonstrates the Metropolis-Hastings algorithm for sampling from discrete probability distributions, with a focus on quantum states for H2 molecules in second quantization representation.

## Key Differences from Continuous Case:
- States are discrete (e.g., (1,0,1,0) for H2 molecular orbitals)
- Proposal distribution must suggest transitions between discrete states
- Acceptance probability calculation remains similar but with discrete state probabilities

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
import collections
import time

# Set up plotting parameters
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 14

## 1. H2 Molecular Orbital States

In second quantization, we represent electronic states using occupation numbers for molecular orbitals. For H2 with a minimal basis, we typically have 4 spin-orbitals:
- 2 spatial orbitals (σg and σu*)
- Each with 2 spin states (α and β)

A state like (1,0,1,0) means:
- Orbital 1 (σgα): occupied (1)
- Orbital 2 (σgβ): empty (0)
- Orbital 3 (σu*α): occupied (1)
- Orbital 4 (σu*β): empty (0)

In [2]:
def generate_all_states(num_orbitals):
    """Generate all possible occupation number states for given number of orbitals"""
    return list(product([0, 1], repeat=num_orbitals))

def state_to_string(state):
    """Convert state tuple to string representation"""
    return ''.join(map(str, state))

def count_electrons(state):
    """Count number of electrons in a given state"""
    return sum(state)

# Generate all possible states for H2 with 4 spin-orbitals
num_orbitals = 4
all_states = generate_all_states(num_orbitals)

# Filter states with exactly 2 electrons (neutral H2)
h2_states = [state for state in all_states if count_electrons(state) == 2]

print(f"Total possible states: {len(all_states)}")
print(f"H2 states with 2 electrons: {len(h2_states)}")
print("\nFirst 10 H2 states:")
for i, state in enumerate(h2_states[:10]):
    print(f"{state_to_string(state)}: {state}")

Total possible states: 16
H2 states with 2 electrons: 6

First 10 H2 states:
0011: (0, 0, 1, 1)
0101: (0, 1, 0, 1)
0110: (0, 1, 1, 0)
1001: (1, 0, 0, 1)
1010: (1, 0, 1, 0)
1100: (1, 1, 0, 0)


## 2. Target Distribution for H2 States

We'll define a target distribution that favors certain electronic configurations based on their energy. Lower energy states will have higher probability.

In [3]:
def calculate_state_energy(state):
    """Calculate energy of a given electronic state
    
    This is a simplified model where:
    - Occupying lower energy orbitals (first two) has lower energy
    - Double occupation of same spatial orbital adds repulsion
    - Excited states have higher energy
    """
    energy = 0.0
    
    # Base energy for each occupied orbital
    orbital_energies = [0.0, 0.0, 1.0, 1.0]  # First two are lower energy
    
    for i, occupied in enumerate(state):
        if occupied:
            energy += orbital_energies[i]
    
    # Add electron-electron repulsion for certain configurations
    # Higher repulsion for electrons in same spatial orbital
    if state[0] == 1 and state[1] == 1:  # Both electrons in σg
        energy += 0.5
    if state[2] == 1 and state[3] == 1:  # Both electrons in σu*
        energy += 0.5
    
    return energy

def target_distribution(state, temperature=1.0):
    """Target distribution using Boltzmann weights"""
    energy = calculate_state_energy(state)
    return np.exp(-energy / temperature)

# Calculate probabilities for all H2 states
state_probabilities = {}
for state in h2_states:
    state_probabilities[state] = target_distribution(state)

# Normalize probabilities
total_prob = sum(state_probabilities.values())
for state in state_probabilities:
    state_probabilities[state] /= total_prob

# Print some example states with their probabilities
print("Example H2 states with their probabilities:")
for state in sorted(state_probabilities.keys(), key=lambda x: state_probabilities[x], reverse=True)[:10]:
    energy = calculate_state_energy(state)
    prob = state_probabilities[state]
    print(f"{state_to_string(state)}: Energy = {energy:.2f}, Probability = {prob:.4f}")

Example H2 states with their probabilities:
1100: Energy = 0.50, Probability = 0.2808
0101: Energy = 1.00, Probability = 0.1703
0110: Energy = 1.00, Probability = 0.1703
1001: Energy = 1.00, Probability = 0.1703
1010: Energy = 1.00, Probability = 0.1703
0011: Energy = 2.50, Probability = 0.0380


## 3. Proposal Distribution for Discrete States

For discrete states, we need to define how to transition between states. We'll use single and double excitations.

In [4]:
def generate_single_excitations(state):
    """Generate all possible single excitations from a given state"""
    excitations = []
    
    # Find occupied and virtual orbitals
    occupied = [i for i, occ in enumerate(state) if occ == 1]
    virtual = [i for i, occ in enumerate(state) if occ == 0]
    
    # Generate all single excitations
    for occ in occupied:
        for virt in virtual:
            new_state = list(state)
            new_state[occ] = 0
            new_state[virt] = 1
            excitations.append(tuple(new_state))
    
    return excitations

def generate_double_excitations(state):
    """Generate all possible double excitations from a given state"""
    excitations = []
    
    # Find occupied and virtual orbitals
    occupied = [i for i, occ in enumerate(state) if occ == 1]
    virtual = [i for i, occ in enumerate(state) if occ == 0]
    
    # Generate all double excitations
    if len(occupied) >= 2 and len(virtual) >= 2:
        for i in range(len(occupied)):
            for j in range(i+1, len(occupied)):
                for k in range(len(virtual)):
                    for l in range(k+1, len(virtual)):
                        new_state = list(state)
                        new_state[occupied[i]] = 0
                        new_state[occupied[j]] = 0
                        new_state[virtual[k]] = 1
                        new_state[virtual[l]] = 1
                        excitations.append(tuple(new_state))
    
    return excitations

def propose_new_state(state, excitation_probability=0.5):
    """Propose a new state through single or double excitation"""
    if np.random.random() < excitation_probability:
        # Single excitation
        excitations = generate_single_excitations(state)
    else:
        # Double excitation
        excitations = generate_double_excitations(state)
    
    if excitations:
        return np.random.choice(excitations)
    else:
        return state  # No excitations possible, return current state

## 4. Discrete Metropolis-Hastings Algorithm

Now we implement the Metropolis-Hastings algorithm for discrete states.

In [5]:
def metropolis_hastings_discrete(initial_state, target_func, proposal_func, 
                               num_samples, temperature=1.0, excitation_probability=0.5):
    """Metropolis-Hastings algorithm for discrete states"""
    samples = []
    current_state = initial_state
    accepted = 0
    
    for i in range(num_samples):
        # Propose new state
        proposed_state = proposal_func(current_state, excitation_probability)
        
        # Calculate acceptance probability
        # For symmetric proposal, q(x*|x) = q(x|x*)
        current_prob = target_func(current_state, temperature)
        proposed_prob = target_func(proposed_state, temperature)
        
        acceptance_prob = min(1, proposed_prob / current_prob)
        
        # Accept or reject
        if np.random.random() < acceptance_prob:
            current_state = proposed_state
            accepted += 1
        
        samples.append(current_state)
    
    acceptance_rate = accepted / num_samples
    return samples, acceptance_rate

# Run the algorithm
np.random.seed(42)
num_samples = 10000
initial_state = (1, 1, 0, 0)  # Start with ground state configuration
temperature = 0.5
excitation_probability = 0.7

samples, acceptance_rate = metropolis_hastings_discrete(
    initial_state, target_distribution, propose_new_state, 
    num_samples, temperature, excitation_probability
)

print(f"Acceptance rate: {acceptance_rate:.2%}")
print(f"Number of samples: {len(samples)}")

ValueError: a must be 1-dimensional

## 5. Analyzing the Results

In [ ]:
# Count frequency of each state
state_counts = collections.Counter(samples)

# Convert to probabilities
sampled_probabilities = {state: count/num_samples for state, count in state_counts.items()}

# Compare with target probabilities
print("Comparison of Target vs Sampled Probabilities:")
print("State\t\tTarget\t\tSampled\t\tDifference")
print("-" * 60)

for state in sorted(sampled_probabilities.keys(), key=lambda x: sampled_probabilities[x], reverse=True):
    target_prob = state_probabilities.get(state, 0)
    sampled_prob = sampled_probabilities[state]
    diff = abs(target_prob - sampled_prob)
    print(f"{state_to_string(state)}\t\t{target_prob:.4f}\t\t{sampled_prob:.4f}\t\t{diff:.4f}")

## 6. Visualization of Results

In [ ]:
# Prepare data for visualization
states = list(sampled_probabilities.keys())
state_labels = [state_to_string(state) for state in states]
sampled_probs = [sampled_probabilities[state] for state in states]
target_probs = [state_probabilities.get(state, 0) for state in states]

# Create bar plot comparison
x = np.arange(len(state_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 8))
bars1 = ax.bar(x - width/2, target_probs, width, label='Target Distribution', alpha=0.7)
bars2 = ax.bar(x + width/2, sampled_probs, width, label='Sampled Distribution', alpha=0.7)

ax.set_xlabel('Electronic States')
ax.set_ylabel('Probability')
ax.set_title('Target vs Sampled Distribution for H2 Electronic States')
ax.set_xticks(x)
ax.set_xticklabels(state_labels, rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 7. Convergence Analysis

In [ ]:
# Convert states to numerical indices for plotting
state_to_index = {state: i for i, state in enumerate(states)}
sample_indices = [state_to_index[state] for state in samples]

# Create trace plot
plt.figure(figsize=(15, 10))

# Trace plot
plt.subplot(2, 2, 1)
plt.plot(sample_indices[:1000])  # Plot first 1000 samples
plt.title('Trace Plot (First 1000 Samples)')
plt.xlabel('Iteration')
plt.ylabel('State Index')
plt.grid(True)

# Running mean of energy
plt.subplot(2, 2, 2)
energies = [calculate_state_energy(state) for state in samples]
running_mean_energy = np.cumsum(energies) / np.arange(1, len(energies) + 1)
plt.plot(running_mean_energy)
plt.title('Running Mean of Energy')
plt.xlabel('Iteration')
plt.ylabel('Mean Energy')
plt.grid(True)

# Autocorrelation of energy
plt.subplot(2, 2, 3)
def autocorrelation(x, max_lag=100):
    n = len(x)
    autocorr = np.zeros(max_lag)
    x_centered = x - np.mean(x)
    
    for lag in range(max_lag):
        if lag == 0:
            autocorr[lag] = 1
        else:
            autocorr[lag] = np.sum(x_centered[:n-lag] * x_centered[lag:]) / np.sum(x_centered**2)
    
    return autocorr

autocorr = autocorrelation(energies)
plt.plot(autocorr)
plt.title('Autocorrelation of Energy')
plt.xlabel('Lag')
plt.ylabel('Autocorrelation')
plt.grid(True)

# Energy distribution
plt.subplot(2, 2, 4)
plt.hist(energies, bins=20, density=True, alpha=0.7, edgecolor='black')
plt.title('Energy Distribution')
plt.xlabel('Energy')
plt.ylabel('Probability Density')
plt.grid(True)

plt.tight_layout()
plt.show()

## 8. Effect of Temperature on Sampling

Temperature controls the exploration vs exploitation trade-off. Higher temperatures lead to more exploration of high-energy states.

In [ ]:
# Test different temperatures
temperatures = [0.1, 0.5, 1.0, 2.0]
results_temp = {}

for temp in temperatures:
    samples_temp, acceptance_rate_temp = metropolis_hastings_discrete(
        initial_state, target_distribution, propose_new_state, 
        num_samples, temp, excitation_probability
    )
    
    state_counts_temp = collections.Counter(samples_temp)
    sampled_probabilities_temp = {state: count/num_samples for state, count in state_counts_temp.items()}
    
    # Calculate average energy
    avg_energy = np.mean([calculate_state_energy(state) for state in samples_temp])
    
    results_temp[temp] = {
        'samples': samples_temp,
        'acceptance_rate': acceptance_rate_temp,
        'probabilities': sampled_probabilities_temp,
        'avg_energy': avg_energy
    }

# Plot results
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for i, temp in enumerate(temperatures):
    ax = axes[i]
    
    # Get top states for this temperature
    temp_probs = results_temp[temp]['probabilities']
    top_states = sorted(temp_probs.keys(), key=lambda x: temp_probs[x], reverse=True)[:10]
    
    state_labels = [state_to_string(state) for state in top_states]
    probs = [temp_probs[state] for state in top_states]
    
    ax.bar(state_labels, probs, alpha=0.7)
    ax.set_title(f'Temperature = {temp}, Acceptance Rate = {results_temp[temp]["acceptance_rate"]:.2%}')
    ax.set_xlabel('State')
    ax.set_ylabel('Probability')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print average energies
print("Average Energy at Different Temperatures:")
print("-" * 40)
for temp in temperatures:
    print(f"Temperature {temp}: {results_temp[temp]['avg_energy']:.4f}")

## 9. Key Differences from Continuous Metropolis-Hastings

### Discrete vs Continuous Implementation:

1. **State Representation**:
   - Continuous: Real-valued vectors (e.g., [1.23, 2.45])
   - Discrete: Occupation number tuples (e.g., (1,0,1,0))

2. **Proposal Distribution**:
   - Continuous: Gaussian or other continuous distributions
   - Discrete: Systematic state transitions (single/double excitations)

3. **State Space**:
   - Continuous: Infinite, unbounded
   - Discrete: Finite, combinatorial (2^n states for n orbitals)

4. **Applications**:
   - Continuous: General Bayesian inference, optimization
   - Discrete: Quantum systems, combinatorial optimization, Ising models

### Advantages of Discrete Approach:
- Exact enumeration of all possible states (for small systems)
- Direct connection to physical quantum systems
- No numerical integration errors
- Clear physical interpretation of state transitions